# How a local LLM answers questions using structured data

**You are here → 00 — How a local AI request works.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| **→ 0. Understand the request (current)** | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| 5. CPU portability (optional) | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| 6. Compatibility study (optional) | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

**Use case:** Help a clinical trial analyst answer questions about a study.
**Persona:** A clinical trial analyst.
**Job:** Analyze a clinical study using structured trial data.

The analyst asks questions such as, “How many enrolled subjects are in each
treatment arm?” The LLM needs more than the question: it needs context—the
database structure, the meaning of “enrolled,” and instructions for writing SQL.
The application then checks the generated SQL and runs it to produce the answer.

Along the way, we'll ask how much database context to send, how to serve the model, and how to
tell whether a quick response is also a useful answer. Our examples use **Qwen3-4B
and Nemotron 3 Nano 4B**, the two models used throughout to **compare how schema context and serving choices affect response time and structured answer quality**.

**Choose how to run notebook 00:**

| Option | What happens | What you need |
|---|---|---|
| **Replay** | Inspect the database and display the supplied experiment results | The notebook environment; no model download or inference server |
| **Live: run from scratch (default)** | Send the selected questions to your running model, execute its SQL, and save a new quality/timing baseline | Model weights, a compatible runtime and hardware, and the local configuration |

For the default Live mode, follow the [from-scratch setup in SETUP.md](../SETUP.md#notebook-00-replay-or-run-from-scratch), start the local server and choose **Run All**.
To explore the supplied results without a server, select **Replay** using the same setup guide.

In this notebook, **we compare full and retrieved schema context**, examine **local serving with llama.cpp and Ollama**, and **evaluate the generated SQL**. Together, these experiments **establish a baseline for response time and SQL answer quality**.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.foundation import inventory, contexts, mac_history
from clinical_trial_mvp.tutorial.foundation_study import FoundationStudy
study = FoundationStudy(ROOT)
print('Chapter 00 mode:', study.mode)
print('Live: run embedding inference and call your local model server.' if study.mode == 'live' else
      'Replay: display the supplied results.' if study.mode == 'replay' else
      'Plan: inspect the configuration.')


Chapter 00 mode: live
Live: run embedding inference and call your local model server.


### The questions we'll work through

We compare **Nemotron 3 Nano 4B (Q4_K_M)** and **Qwen3-4B (Q4_K_M)**
using full and retrieved schema context. The local-serving comparison uses the
same **Nemotron** GGUF file through llama.cpp and Ollama. The structured-answer
example also uses Nemotron through llama.cpp.

| Section | What we want to understand |
|---|---|
| 1. Workload | What structured data and questions must the application handle? |
| 2. Schema context | How do full and retrieved schema affect Nemotron and Qwen's response time and result quality? |
| 3. Local serving | How does the same Nemotron model behave through llama.cpp and Ollama? |
| 4. Structured answer | How does Nemotron's generated SQL become a structured answer with a quality check? |
| 5. Establish the baseline | What quality and timing baseline do these comparisons establish? |

## 1. The application and its data

Our application answers questions about subjects, sites, visits and adverse
events in a clinical trial. The database is synthetic; it contains no personally identifiable information (PII).
**DuckDB** is the SQL database engine that executes the generated queries.

Here's the path we'll follow:

**Question → schema context → model plan and SQL → validation → DuckDB → result comparison.**

A *schema* describes tables, columns and relationships. A *plan* states what the
answer should count or group before the model writes SQL. Validation checks
whether the generated query is allowed and executable; result comparison checks
its rows against a reference query.

Run the next cell to inspect the ten tables. This notebook reads the supplied
database without changing it. Using the same rows lets us compare the model's
query with the reference consistently.

The evaluation set contains **30 questions**, including **23 with reference SQL**.
The optimization chapters use four questions from that set for shorter experiments.
Repeating those four questions helps measure timing.


In [2]:
database = inventory(ROOT)
display(JSON(database))


<IPython.core.display.JSON object>

## 2. Give the model the context it needs

To write SQL, the model needs table names, columns, joins and the definitions
behind the question. We can prepare that context in two ways:

- **Full schema:** include all the available schema descriptions.
- **Retrieved schema:** select the descriptions that appear relevant to this
  question, using embeddings to find related text.

Both approaches send text to the generation model. Retrieval reduces the amount
of text, but it may also leave out a definition or join the answer needs.

The next cell displays **both prompts for the same question**. Compare the full
prompt with the retrieved prompt: what was kept, and what was removed? The
displayed lengths count characters, not model tokens.

First inspect the fixed schema selection used by the baseline comparison.
Then run the embedding cell below to see how a fresh selection is produced.
The timing comparison measures answer generation and application work;
it excludes schema retrieval.


In [3]:
context = contexts(ROOT)
display(JSON({'note': context['note'], 'cases': [
    {key: value for key, value in row.items() if key not in ('full_prompt', 'retrieved_prompt')}
    for row in context['rows']]}))
display(JSON({'full_prompt': context['rows'][0]['full_prompt'],
              'retrieved_prompt': context['rows'][0]['retrieved_prompt']}))


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

### Invoke the embedding model

**Nemotron-3-Embed-1B-BF16** selects schema context for both SQL-generating
models. It runs locally through Sentence Transformers, independently of the
OpenAI-compatible server used for plan and SQL generation.

The next cell loads the embedding model and calls `embedding_model.encode()`.
The retriever encodes schema descriptions with a **`passage: `** prefix and the
question with a **`query: `** prefix. It ranks their normalized vectors by
similarity, selects up to 12 facts, then adds required columns and join paths.
Read the selected facts and schema to see what the generation model could receive.

In **Live**, this is fresh embedding inference. Follow the
[embedding setup](../SETUP.md#notebook-00-embedding-model) first.
In **Replay**, this cell is skipped. The measured comparisons continue to use
the fixed retrieval selection displayed above so their inputs remain consistent;
this demonstration does not replace that selection or enter the latency table.


In [4]:
# Fresh local embedding inference; no chat-completions API is used here.
if study.mode == 'live':
    from sentence_transformers import SentenceTransformer
    from clinical_trial_mvp.data.embedding_retrieval import SchemaRetriever

    embedding_path = Path(study.config.get(
        'embedding_model_path', 'models/Nemotron-3-Embed-1B-BF16')).expanduser()
    if not embedding_path.is_absolute():
        embedding_path = ROOT / embedding_path
    if not embedding_path.is_dir():
        raise FileNotFoundError('Download the embedding model first: see SETUP.md#notebook-00-embedding-model')
    embedding_model = SentenceTransformer(
        str(embedding_path), device='cpu', local_files_only=True, trust_remote_code=False)

    class NotebookEmbedder:
        def encode(self, texts):
            # Inputs already carry passage: or query: prefixes from SchemaRetriever.
            return embedding_model.encode(
                texts, batch_size=8, normalize_embeddings=True,
                show_progress_bar=False).tolist()

    # Construction embeds schema facts; retrieve() embeds the question.
    retriever = SchemaRetriever(NotebookEmbedder(), top_k=12, max_seed_tables=4)
    embedding_question = context['rows'][0]['question']
    selected_context = retriever.retrieve(embedding_question)
    display(Markdown('**Question:** ' + embedding_question))
    display(JSON({
        'embedding_model': 'nvidia/Nemotron-3-Embed-1B-BF16',
        'selected_tables': selected_context.tables,
        'ranked_facts': [
            {'fact': fact.text, 'similarity': selected_context.scores[fact.fact_id]}
            for fact in sorted(
                (fact for fact in retriever.facts if fact.fact_id in selected_context.scores),
                key=lambda fact: selected_context.scores[fact.fact_id], reverse=True)],
    }))
    print('Retrieved schema text:')
    print(selected_context.prompt)
else:
    print('Select Live mode to invoke the embedding model; Replay uses the supplied selection.')


/tmp/local-ai-inference-clean-20261109/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 7238.48it/s]


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


**Question:** How many subjects are Enrolled or Randomized by treatment arm?

<IPython.core.display.JSON object>

Retrieved schema text:
DuckDB clinical trial schema (retrieved facts):

Table: subjects
- subject_id VARCHAR: Primary synthetic subject identifier. Treat as private.
- treatment_arm VARCHAR: Assigned treatment arm or Not Assigned.
- screening_status VARCHAR: Screening/enrollment status: Enrolled, Randomized, Screen Failed, or Withdrawn. Examples: Enrolled, Randomized, Screen Failed, Withdrawn.
- enrollment_date DATE: Subject enrollment date when available.

Relationships:

Clinical business definitions:
- Enrolled subjects: screening_status IN ('Enrolled', 'Randomized'); count subjects, not child rows.

Privacy: subject_id is a private join key. Prefer aggregates; the MCP server pseudonymizes subject_id in displayed results.


### Make the first model API call

Now send the first question and its full schema context to the running model.
The cell below shows the **OpenAI-compatible request**, executes
`POST /v1/chat/completions`, and prints the model's new response: an answer plan.
It uses **Nemotron 3 Nano 4B Q4_K_M through llama.cpp** with the default local setup.
The endpoint and served model name come from your local configuration.

Run the [local-server setup](../SETUP.md#notebook-00-replay-or-run-from-scratch)
first. **Live** executes this request; **Replay** skips it. This introductory
request is separate from the measured comparisons below. The application then
uses the plan in a second model call to generate SQL.


In [5]:
# First model inference: an OpenAI-compatible chat-completions request.
import json
import urllib.request
from clinical_trial_mvp.agent.analyst import (
    PLAN_CHAT_SYSTEM_MESSAGE, PLAN_JSON_SCHEMA, build_plan_prompt,
)
from clinical_trial_mvp.serving.benchmark import Workload, prepare
from clinical_trial_mvp.serving.client import Generation, OpenAIClient

if study.mode == 'live':
    endpoint = study.config['servers'][study.config['context_server']]
    if endpoint['runtime'] not in {'llama_cpp', 'vllm', 'sglang'}:
        raise ValueError('Select an OpenAI-compatible endpoint for this example; see SETUP.md.')
    cases, schema_contexts, _, _ = prepare(
        ROOT, Workload(profile='full'), study.config['case_ids'])
    first_question = cases[0]['question']
    schema_text, _, definitions = schema_contexts[first_question]
    messages = [
        {'role': 'system', 'content': PLAN_CHAT_SYSTEM_MESSAGE},
        {'role': 'user', 'content': build_plan_prompt(first_question, schema_text, definitions)},
    ]
    # The same request settings and local-only transport as the benchmark.
    api = OpenAIClient(endpoint['base_url'], endpoint['model'],
        Generation(max_tokens=study.config['max_tokens'], seed=42),
        request_profile=endpoint['runtime'])
    api_url = api.base_url + '/chat/completions'
    payload = api.request_payload(messages, PLAN_JSON_SCHEMA)
    payload['stream'] = False
    payload.pop('stream_options', None)
    display(Markdown('**Question:** ' + first_question))
    print('POST', api_url)
    display(JSON(payload))

    # This line contacts the running model server. It does not load a saved answer.
    request = urllib.request.Request(
        api_url, data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'}, method='POST')
    with api.opener(request, timeout=180) as response:
        api_response = json.load(response)
    choice = api_response['choices'][0]
    if choice.get('finish_reason') != 'stop':
        raise RuntimeError('Model response did not finish normally: ' + str(choice.get('finish_reason')))
    model_reply = choice['message']['content']
    if not isinstance(model_reply, str) or not model_reply.strip():
        raise RuntimeError('The model returned an empty answer plan.')
    print('Model response:')
    print(model_reply)
    if api_response.get('usage'):
        display(JSON(api_response['usage']))
else:
    print('Select Live mode and start the server to execute this API example (see SETUP.md).')


**Question:** How many subjects are Enrolled or Randomized by treatment arm?

POST http://127.0.0.1:8080/v1/chat/completions


<IPython.core.display.JSON object>

Model response:
{
  "population": "subjects where screening_status IN ('Enrolled','Randomized') AND treatment_arm IS NOT NULL",
  "measure": "COUNT(DISTINCT subject_id)",
  "grouping": "treatment_arm",
  "time_window": "none",
  "filters": "screening_status IN ('Enrolled','Randomized') AND treatment_arm IS NOT NULL",
  "ranking": "none",
  "needs_clarification": false,
  "clarification_question": ""
}


<IPython.core.display.JSON object>

### Compare full-schema and retrieved-schema answers

Now let's see what happened when each model answered using those two kinds of
context. This experiment used **Nemotron 3 Nano 4B and Qwen3-4B**, each in
**Q4_K_M GGUF** format through **llama-cpp-python 0.3.35**, on an **Apple M4 Pro
CPU** with GPU offload disabled. GGUF is the model-file format used by llama.cpp;
Q4_K_M identifies the quantized weight representation.

Both models receive the same context selected by Nemotron Embed. This keeps retrieval fixed for the comparison

Each model/context configuration in Replay answered the 30-question set once.
The next table compares result matches on the 23 questions with reference SQL and
**application latency**: planning, generation, checks and any repair. This is
the time to work through an answer, rather than the time to its first token.

**In Live:** the next cell runs both schema profiles against `context_server`
from your local config. The default selects four questions; choose all 30 IDs
to run the complete set. The fresh table includes application quality and direct
request timing, labeled separately.


In [6]:
if study.mode == 'replay':
    from clinical_trial_mvp.tutorial.foundation import context_results_table
    history = mac_history(ROOT)
    print('Loading results.')
    display(Markdown(context_results_table(history['context'])))
else:
    context_runs = study.context_rows()
    display(Markdown(study.table(context_runs)))


**Fresh measurements**

| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| Baseline/context/nemotron/full/c1 | 2/4 | 2/4 | Unreviewed; 4 total | 2426 | 6.36 | 8.82/13.26 | 42.72 | 5.61 GiB |
| Baseline/context/nemotron/retrieved/c1 | 2/4 | 1/4 | Unreviewed; 4 total | 481 | 5.29 | 7.16/12.20 | 48.38 | 8.16 GiB |

### What changes with retrieved schema?

In this question set, both models had lower application latency with retrieved
schema. Reference-result matches decreased for Nemotron and increased slightly
for Qwen.

**Takeaway:** Reducing context can save time, but the model still needs the
right definitions and joins to produce a useful structured answer. Read latency
and result matches together when choosing how much context to provide.

See [the schema-comparison settings and detailed results](../baseline/mvp1/experiment-tracker/MVP1_Q4_30_CASE_REPORT.md).


## 3. Serve the same model through llama.cpp and Ollama

Once we've chosen a model and prompt, the application needs a way to call the
model. **llama.cpp** gives us direct control over model execution. **Ollama**
provides model management and an API that the application can call.

Here we hold the **Nemotron 3 Nano 4B Q4_K_M GGUF file** fixed and compare those
two serving paths on an **Apple M4 Pro with Metal GPU offload enabled**.
Prompts, seed and output cap match; the measurements report no cached prompt
tokens. There are three measured requests per prompt for each runtime.

In Replay, run the next cell to compare **time to first token** and **complete request
time**. This measures a model-generation request. Section 2 measured the broader
application path on CPU, so the two sections answer different timing questions.

You can follow how the application calls each runtime in the
[runtime adapters](../src/clinical_trial_mvp/tutorial/adapters.py) and
[serving harness](../src/clinical_trial_mvp/serving/benchmark.py).

**In Live:** this section measures only the running llama.cpp/Ollama endpoints
listed in `local_servers`. Select the local serving endpoints you want to compare in your configuration.
To compare the two paths, supply metadata for the same GGUF artifact and matching
launch settings; these optional endpoint integrations still need fresh hardware validation.


In [7]:
if study.mode == 'replay':
    from clinical_trial_mvp.tutorial.foundation import local_serving_results_table
    print('Loading results.')
    display(Markdown(local_serving_results_table(history['serving'])))
else:
    local_runs = study.local_rows()
    display(Markdown(study.table(local_runs)))


**Fresh measurements**

| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| Baseline/local-serving/nemotron/full/c1 | 2/4 | 2/4 | Unreviewed; 4 total | 2426 | 5.52 | 8.45/13.27 | 41.35 | 8.35 GiB |

### Compare the serving paths

With selected schema, llama.cpp and Ollama produced matching outputs with similar
first-token and request-completion times. The table also includes full-schema
requests so you can compare the effect of context length within each runtime.

**Takeaway:** Runtime controls, API convenience and model management can guide
this serving choice alongside timing. These measurements cover individual model
requests; evaluate the resulting structured answers separately.

Each row summarizes three measured requests per prompt and runtime. See
[the serving setup, prompts and detailed results](../baseline/mvp2/experiment-tracker/mvp2/serving/MILESTONE_REPORT.md).


## 4. Follow one question through to a structured answer

Let's connect the pieces with **Nemotron 3 Nano 4B Q4_K_M through llama.cpp**
on an **Apple M4 Pro CPU**. The next cell shows an actual question, the generated
SQL, its aggregate result rows and the reference-result check.

First identify what the question asks. Check whether the SQL keeps its filters
and grouping and counts the intended subjects. Then read the structured answer
and its quality checks. A query executing successfully is one check; returning
the expected result is another.

In Replay, the example comes from the supplied local-model evaluation. In Live,
the cell displays the full-schema application result collected in your session.

**Question → schema context → generated SQL → execution → structured answer → quality check.**


In [8]:
from clinical_trial_mvp.tutorial.foundation import local_answer_example, structured_answer_markdown
if study.mode == 'replay':
    answer = local_answer_example(ROOT)
else:
    answer = study.answer()
display(Markdown(structured_answer_markdown(answer)))


### Question
eval_001

### Generated SQL
```sql
SELECT treatment_arm, COUNT(DISTINCT subject_id) AS enrolled_subjects FROM subjects WHERE screening_status IN ('Enrolled','Randomized') GROUP BY treatment_arm;
```

### Structured answer
| Treatment arm | Enrolled subjects |
|---|---|
| Placebo | 44 |
| Investigational Drug Low Dose | 48 |
| Investigational Drug High Dose | 40 |
| Standard of Care | 38 |

### Quality check
| Check | Result |
|---|---|
| SQL executed | Passed |
| Reference-result match | Passed |

## 5. Establish the baseline

The table below brings together this notebook's schema-context, local-serving
and structured-answer results. Read each comparison's quality and timing together.


In [9]:
from clinical_trial_mvp.tutorial.foundation import baseline_results_summary
baseline_history = history if study.mode == 'replay' else None
display(Markdown(baseline_results_summary(study, baseline_history, answer)))


| Baseline | Schema context | Reference-result matches | Median application latency (s) | Median first-token time (s) | Median request completion (s) |
|---|---|---:|---:|---:|---:|
| context/nemotron/full/c1 | full | 2/4 | 38.037 | 6.359 | 8.824 |
| context/nemotron/retrieved/c1 | retrieved | 1/4 | 30.898 | 5.293 | 7.161 |
| local-serving/nemotron/full/c1 | full | 2/4 | 33.269 | 5.523 | 8.447 |
| Structured-answer example | full | Matched (one question) | 26.042 | — | — |

Application latency includes planning and SQL checks; request timing covers the model-generation call.
A dash indicates a metric not measured in that comparison.

**Result summary**

The table brings together the context and local-serving results from this run. Read reference-result matches alongside response times to assess each configuration. The worked structured answer executed successfully and matched its reference result.